# Touch Data Flagging System: Visual Guide

This notebook provides a visual explanation of how our touch data processing pipeline identifies and flags sequence issues.

## 1. Valid Touch Sequence

A valid touch sequence represents a complete finger interaction from touch to lift. It follows a specific pattern:

### Coloring Data Valid Sequence

```
┌─────────┐     ┌─────────┐     ┌─────────┐     ┌─────────┐
│  Began  │ ──► │  Moved  │ ──► │  Moved  │ ──► │  Ended  │
└─────────┘     └─────────┘     └─────────┘     └─────────┘
    │               │               │               │
    ▼               ▼               ▼               ▼
Finger down     Movement        Movement        Finger up
```

**Required Elements:**
- **Begin Event**: `touchPhase = "Began"` - When finger first touches screen
- **Middle Events**: `touchPhase = "Moved"` or `"Stationary"` - As finger moves or stays still
- **End Event**: `touchPhase = "Ended"` or `"Canceled"` - When finger is lifted

## 2. Sequence ID Assignment

Each touch event is assigned a `seqId` (sequence identifier) that groups related events together.

### How Sequence IDs Work

```
┌─────────────────────────────────────────────────────────┐
│ fingerId = 0                                            │
├───────────┬───────────┬───────────┬───────────┬─────────┤
│ seqId = 1 │ seqId = 2 │ seqId = 3 │ seqId = 4 │   ...   │
└───────────┴───────────┴───────────┴───────────┴─────────┘

┌─────────────────────────────────────────────────────────┐
│ fingerId = 1                                            │
├───────────┬───────────┬───────────┬───────────┬─────────┤
│ seqId = 1 │ seqId = 2 │ seqId = 3 │ seqId = 4 │   ...   │
└───────────┴───────────┴───────────┴───────────┴─────────┘
```

**Key Points:**
- Each finger (`fingerId`) has its own independent sequence numbering
- Sequences are numbered sequentially starting from 1
- `seqId = 0` is reserved for events that don't belong to any valid sequence
- A new sequence begins only after the current sequence has properly ended

### Multi-Finger Example

```
Time ─────────────────────────────────────────────────────────────►

Finger 0: [Sequence 1]───[Sequence 2]───[Sequence 3]
           B→M→M→E        B→M→E          B→M→M→M→E

Finger 1:      [Sequence 1]───────[Sequence 2]
                B→M→M→M→E          B→M→E
```

**Note:** Each finger maintains its own sequence numbering, allowing for multi-touch interactions.

## 3. Flag Visual Reference Guide

Our system applies flags to sequences that don't follow the expected pattern. Here's a visual guide to each flag:

### `missing_Began` Flag

```
      ●───────●───────●───────●───────●───────●───────●
      │       │       │       │       │       │       │
      │       │       │       │       │       │       │
    Moved   Moved   Moved   Moved   Moved   Moved   Ended
    
    ⚠️ missing_Began
```

**Plain English:** The sequence is missing its starting point.

**Common Cause:** Touch began outside the active sensing area of the screen.

### `missing_Ended` Flag

```
  ●───────●───────●───────●───────●───────●───────●
  │       │       │       │       │       │       │
  │       │       │       │       │       │       │
Began   Moved   Moved   Moved   Moved   Moved   Moved
  
  ⚠️ missing_Ended
```

**Plain English:** The sequence never properly ended.

**Common Cause:** User lifted their finger outside the active sensing area.

### `multiple_end_events` Flag

```
  ●───────●───────●───────●───────●───────●───────●───────●
  │       │       │       │       │       │       │       │
  │       │       │       │       │       │       │       │
Began   Moved   Moved   Ended   Moved   Moved   Moved   Ended
  
  ⚠️ multiple_end_events
```

**Plain English:** The sequence has more than one ending event.

**Common Cause:** Touch sensor incorrectly detected finger lift and then redetected the same touch.

### `sequence_interrupted` Flag

```
  ●───────●───────●───────●───────●
  │       │       │       │       │
  │       │       │       │       │
Began   Moved   Moved   Moved   Began (new sequence)
  
  ⚠️ sequence_interrupted
```

**Plain English:** A new sequence started before the current one ended.

**Common Cause:** Hardware issue causing "phantom" begin events.

### `orphaned_events` Flag

```
  ●───────●───────●───────●       ●───────●───────●───────●
  │       │       │       │       │       │       │       │
  │       │       │       │       │       │       │       │
Began   Moved   Moved   Ended   Moved   Moved   Moved   Began (new sequence)
                              │       │       │
                              └───────┴───────┘
                               Orphaned events
  
  ⚠️ orphaned_events
```

**Plain English:** Touch events occurred between sequences, not belonging to any sequence.

**Common Cause:** Touch sensor detected movement after the finger was lifted.

### `short_duration` Flag

```
  ●───●
  │   │
  │   │
Began Ended
  
  ⚠️ short_duration (< 0.01 seconds)
```

**Plain English:** The touch sequence was extremely brief (less than 10ms).

**Common Cause:** User accidentally brushed against the screen.

### `too_few_points` Flag

```
  ●───────●
  │       │
  │       │
Began   Ended
  
  ⚠️ too_few_points (< 3 points)
```

**Plain English:** The sequence has fewer than 3 touch events.

**Common Cause:** User performed a quick tap rather than a drag or swipe.

### `sequence_gap` Flag

```
  ●───────●         ●───────●───────●
  │       │         │       │       │
  │       │         │       │       │
Began   Moved     Moved   Moved   Ended
            \     /
             \   /
              GAP
             >100ms
  
  ⚠️ sequence_gap
```

**Plain English:** There was a significant time gap (>100ms) between events in a sequence.

**Common Cause:** Application experienced performance issues or frame drops.

### `improper_sequence_order` Flag

```
  ●───────●───────●───────●
  │       │       │       │
  │       │       │       │
Began   Ended   Moved   Moved
  
  ⚠️ improper_sequence_order
```

**Plain English:** Touch events don't follow the expected order (Began → Moved/Stationary → Ended).

**Common Cause:** Touch events were processed out of order due to threading or timing issues.

## 4. Flag Precedence Hierarchy

When conflicts occur between flags, the system resolves them using this priority hierarchy:

```
┌─────────────────────┐
│  orphaned_events    │  Priority 1 (Highest)
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│sequence_interrupted │  Priority 2
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│ multiple_end_events │  Priority 3
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│    missing_Began    │  Priority 4
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│    missing_Ended    │  Priority 5
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│improper_sequence_order│ Priority 6
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│   short_duration    │  Priority 7
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│    too_few_points   │  Priority 8
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│     has_canceled    │  Priority 9
└──────────┬──────────┘
           ▼
┌─────────────────────┐
│    sequence_gap     │  Priority 10 (Lowest)
└─────────────────────┘
```

### Conflict Resolution Examples

| Conflicting Flags | Resolution | Reason |
|-------------------|------------|--------|
| `missing_Ended` + `multiple_end_events` | Keep `multiple_end_events` | Having multiple end events is more specific than saying an end is missing |
| `missing_Began` + `orphaned_events` | Keep `orphaned_events` | Orphaned events explain why a beginning might be missing |
| `sequence_interrupted` + `missing_Ended` | Keep `sequence_interrupted` | An interruption explains why an end might be missing |